# FIFO回环与仿射计算：双通路验收

领航者7020，PYNQ 2.7。原FIFO回环保留，仿射计算使用独立DMA。

| 通路 | DMA | 地址 | 计算 |
|---|---|---|---|
| FIFO | axi_dma_0 | 0x40400000 | 原样返回 |
| 仿射 | axi_dma_affine | 0x40410000 | uint32：y=(3x+7) mod 2^32 |

步骤和重建方法见 `references/dual-stream-board.md`。默认仅检查环境；运行硬件时将参数改为True。


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
assert (ROOT/'scripts/pynq_debug.py').is_file(), '在技能包根目录打开本Notebook'
sys.path.insert(0, str(ROOT/'scripts'))
sys.path.insert(0, str(ROOT/'examples/affine_u32'))
from pynq_debug import environment
from board_suite import run_suite
print(json.dumps(environment(), indent=2))


## 1. 配置

bit/hwh来自同次双通路构建。启用后下载一次位流，在同一配置下交替验证两条通路，最后重新加载并复测。


In [ ]:
RUN_HARDWARE = False
BITFILE = ROOT/'overlays/affine_dual.bit'
FIFO_DMA = 'axi_dma_0'
AFFINE_DMA = 'axi_dma_affine'
COMPUTE_IP = 'affine_u32_stream_0'
print('Hardware enabled:', RUN_HARDWARE)
print('Bit/hwh present:', BITFILE.is_file(), BITFILE.with_suffix('.hwh').is_file())


## 2. 执行验收

仿射24组；FIFO的uint32测试24组、字节边界3组；随后检查错误参考、超长请求和两条通路重新加载。每次结果保存JSON与输入/参考/实际输出NPZ。

遇到缓冲区隔离时保持内核和缓冲区，先恢复开发板。正常完成后下方显示每条通路的通过数量及证据目录。


In [ ]:
if RUN_HARDWARE:
    summary, evidence_directory = run_suite(BITFILE, fifo_dma=FIFO_DMA,
        affine_dma=AFFINE_DMA, compute_ip=COMPUTE_IP, directory=ROOT/'reports')
    print(json.dumps({k:summary[k] for k in
        ('status','fifo_cases_passed','affine_cases_passed','negative_checks','reload_checks')},
        indent=2))
    print('Evidence directory:', evidence_directory)
else:
    print('READY: 设置 RUN_HARDWARE=True 后执行实板验收')
